In [ ]:
import os
import json
import time
import requests
import urllib3
from dotenv import load_dotenv
from google_auth_oauthlib.flow import InstalledAppFlow

In [ ]:
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

In [ ]:
load_dotenv()
MIO_CLIENT_ID     = os.getenv("GOOGLE_CLIENT_ID")
MIO_CLIENT_SECRET = os.getenv("GOOGLE_CLIENT_SECRET")
MAPS_API_KEY      = os.getenv("GOOGLE_MAPS_API_KEY")

In [ ]:
def ottieni_credenziali():
    cfg = {
        "installed": {
            "client_id": MIO_CLIENT_ID,
            "client_secret": MIO_CLIENT_SECRET,
            "auth_uri": "https://accounts.google.com/o/oauth2/auth",
            "token_uri": "https://oauth2.googleapis.com/token",
            "redirect_uris": ["http://localhost"]
        }
    }
    flow = InstalledAppFlow.from_client_config(
        cfg, scopes=["https://www.googleapis.com/auth/business.manage"]
    )
    flow.authed_session = requests.Session()
    flow.authed_session.verify = False
    return flow.run_local_server(port=0)

In [ ]:
def main():
    print("--- 🚀 ESTRAZIONE GLOBALE: DATI + ATTRIBUTI + FOTO ---")

    if not MAPS_API_KEY:
        print("⚠️  ATTENZIONE: GOOGLE_MAPS_API_KEY non trovata nel .env. Le foto e il fallback latlng verranno saltati.")

    creds = ottieni_credenziali()
    headers = {"Authorization": f"Bearer {creds.token}", "Content-Type": "application/json"}

    print("\n⏳ 1. Recupero l'elenco dei gruppi aziendali...")
    res_acc = requests.get(
        "https://mybusinessaccountmanagement.googleapis.com/v1/accounts",
        headers=headers, verify=False
    )
    accounts = res_acc.json().get('accounts', [])
    print(f"📂 Trovati {len(accounts)} gruppi.")

    tutti_i_locali_completi = []

    # Maschera dati base (inclusi metadata per avere il placeId)
    read_mask = (
        "name,title,storeCode,storefrontAddress,latlng,phoneNumbers,"
        "categories,websiteUri,regularHours,openInfo,profile,metadata,"
        "moreHours,labels"
    )

    for acc in accounts:
        acc_name = acc['name']
        acc_title = acc.get('accountName', 'Senza nome')
        print(f"\n🔎 Esploro il gruppo: {acc_title} ({acc_name})")

        page_token = None

        while True:
            url = f"https://mybusinessbusinessinformation.googleapis.com/v1/{acc_name}/locations?readMask={read_mask}"
            if page_token:
                url += f"&pageToken={page_token}"

            res = requests.get(url, headers=headers, verify=False)
            if res.status_code != 200:
                print(f"   ❌ Errore recupero locali: {res.status_code} — {res.text[:200]}")
                break

            data_pagina = res.json()
            locali_pagina = data_pagina.get('locations', [])

            # --- FASE DI ARRICCHIMENTO PER OGNI LOCALE ---
            for i, loc in enumerate(locali_pagina):
                titolo = loc.get('title', 'Senza Nome')
                nome_api = loc.get('name')
                place_id = loc.get("metadata", {}).get("placeId", "")
                print(f"   ➕ [{i+1}/{len(locali_pagina)}] Arricchisco: {titolo}...")

                loc['_gruppo'] = acc_title
                loc['_account'] = acc_name

                # --- A. ATTRIBUTI ---
                url_attr = f"https://mybusinessbusinessinformation.googleapis.com/v1/{nome_api}/attributes"
                res_attr = requests.get(url_attr, headers=headers, verify=False)

                attributi_piatti = {}
                if res_attr.status_code == 200:
                    items = res_attr.json().get("attributes", [])
                    for item in items:
                        attr_id = item.get("name", "").replace("attributes/", "")
                        value_type = item.get("valueType", "")

                        if value_type in ("BOOL", "ENUM"):
                            attributi_piatti[attr_id] = item.get("values", [None])[0]
                        elif value_type == "REPEATED_ENUM":
                            set_vals = item.get("repeatedEnumValue", {})
                            attributi_piatti[attr_id] = {
                                "set": set_vals.get("setValues", []),
                                "unset": set_vals.get("unsetValues", [])
                            }
                        elif value_type == "URL":
                            urls = item.get("urlValues", [])
                            attributi_piatti[attr_id] = urls[0].get("url", "") if urls else ""
                        else:
                            attributi_piatti[attr_id] = item.get("values")
                else:
                    print(f"      ⚠️  Attributi non recuperati: {res_attr.status_code}")

                loc['attributi'] = attributi_piatti

                # --- B. FOTO + FALLBACK LATLNG (Tramite Places API) ---
                foto_list = []

                # FIX: check robusto — gestisce sia campo assente che oggetto vuoto {}
                latlng = loc.get("latlng")
                latlng_mancante = not latlng or not latlng.get("latitude")

                if MAPS_API_KEY and place_id:
                    url_place = f"https://places.googleapis.com/v1/places/{place_id}"
                    headers_maps = {
                        "X-Goog-Api-Key": MAPS_API_KEY,
                        "X-Goog-FieldMask": "photos,location"
                    }
                    try:
                        # FIX: verify=False aggiunto per compatibilità con firewall aziendale
                        res_place = requests.get(url_place, headers=headers_maps, verify=False)

                        if res_place.status_code == 200:
                            place_data = res_place.json()

                            # Fallback latlng
                            if latlng_mancante:
                                place_location = place_data.get("location", {})
                                if place_location:
                                    loc["latlng"] = {
                                        "latitude": place_location.get("latitude"),
                                        "longitude": place_location.get("longitude")
                                    }
                                    print(f"      📍 latlng recuperato via Places API.")
                                else:
                                    print(f"      ⚠️  latlng assente anche da Places API per {titolo}")

                            # Foto
                            for photo in place_data.get("photos", [])[:5]:
                                photo_name = photo.get("name", "")
                                if photo_name:
                                    foto_list.append({
                                        "url": (
                                            f"https://places.googleapis.com/v1/{photo_name}/media"
                                            f"?maxWidthPx=1200&key={MAPS_API_KEY}"
                                        ),
                                        "width": photo.get("widthPx"),
                                        "height": photo.get("heightPx"),
                                        "attribution": photo.get("authorAttributions", [{}])[0].get("displayName", "")
                                    })
                        else:
                            print(f"      ❌ Places API error {res_place.status_code} per {titolo}: {res_place.text[:200]}")

                    except requests.exceptions.SSLError as e:
                        print(f"      ❌ SSL Error su Places API per {titolo}: {e}")
                    except Exception as e:
                        print(f"      ❌ Errore generico su Places API per {titolo}: {e}")

                elif latlng_mancante and not place_id:
                    print(f"      ⚠️  latlng assente e placeId mancante per {titolo} — impossibile recuperare coordinate")

                loc['foto'] = foto_list

                # Pausa per non intasare l'API di Maps
                time.sleep(0.2)

            # Aggiungiamo alla lista globale
            tutti_i_locali_completi.extend(locali_pagina)

            # FIX: leggiamo nextPageToken da data_pagina, non da res.json() che rieseguirebbe la chiamata
            page_token = data_pagina.get('nextPageToken')
            if not page_token:
                break

    # Salvataggio
    output_file = "database_completo.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(tutti_i_locali_completi, f, indent=4, ensure_ascii=False)

    print(f"\n🎉 OPERAZIONE COMPLETATA!")
    print(f"📦 Locali salvati: {len(tutti_i_locali_completi)}")
    print(f"💾 File creato: {output_file}")


if __name__ == '__main__':
    main()